# Notebook 2b — MTEB crash course: the benchmark, the library, what the roadmap takes from it

A side notebook, not a phase deliverable. Notebook 2 ended on a table: three embedding models on
SciFact, nDCG@10, next to the published scores. That table came out of one library call, `mteb`,
and the roadmap keeps leaning on it: the phase 1 model comparison, the number the phase 2 harness
must reproduce, the `title + " " + text` convention adopted on 2026-11-02, the BEIR datasets of
phase 3. This notebook opens the black box: **what MTEB is** (a paper, a leaderboard, a library),
**what the library does** when it scores a model on SciFact, step by step in its own source, and
**which parts the testbench needs** — and which it does not. Read notebooks 0 and 2 first.

Everything that belongs to the project is imported from `src/` and run as is ([`run_scifact`](../../src/llm_testbench/eval/embeddings_bench.py#L167) and
its neighbours). The third-party code is read where it lives: the links into `mteb` are GitHub
permalinks pinned to the installed version (2.20.5, files checked identical to the tag); the same
files sit in `.venv/lib/python3.12/site-packages/mteb/` if you prefer the editor.

**Modes.** `mteb` belongs to the `bench` dependency group (`uv sync --group bench`; torch and
sentence-transformers come with it), which the CI does not install: the cells that need it say
so and skip. Sample mode (`LLM_TESTBENCH_SAMPLE=1`) reads the test fixtures and the archived
report and loads no model; full mode runs the configured model on the real SciFact (about 20 s
for all-MiniLM-L6-v2 on Apple Silicon, 2 min for bge-base-en-v1.5, both already in the Hugging
Face cache after `make bench-embeddings`). No API key is needed anywhere in this notebook.

## Part 1 — The problem: "which embedding model?" has no answer without a protocol

An **embedding model** turns a text into a vector of numbers so that texts close in meaning get
close vectors (notebook 2, part 3). Semantic search is built on it: embed the documents once,
embed the question, return the nearest vectors. Every provider sells one — OpenAI, Cohere,
Voyage, Mistral — and Hugging Face hosts thousands of open ones. The model cards all say "state
of the art".

The naive way to choose is to read the cards. It fails for three reasons:

- **Numbers that are not comparable.** A card quotes an average over tasks the author chose,
  computed by the author's code, on a version of the dataset nobody names. Two cards, two
  protocols, no comparison.
- **Numbers that are not yours.** An average over web search, duplicate detection and sentiment
  classification says little about finding the abstract that supports a medical claim.
- **Numbers you cannot check.** If your own measurement does not match the card, you cannot
  tell whether the card, your code or your data is wrong.

What is needed is what the testbench needs everywhere (`CLAUDE.md` §1): a **benchmark** — a
public dataset with annotated answers, a fixed metric, a reference implementation anybody can
rerun — and the discipline of comparing against it. For embedding models that benchmark exists.

**MTEB is three things under one name.**

1. **A paper and a collection of tasks.** *MTEB: Massive Text Embedding Benchmark* (Muennighoff
   et al., 2022): 56 datasets grouped in 8 **task types** — retrieval, reranking,
   classification, clustering, pair classification, semantic textual similarity, summarization,
   bitext mining — each with a main metric, so that one model gets one score per task and one
   average. The 2025 extension (MMTEB) brought it to hundreds of tasks in 250+ languages; the
   English set used for ranking today is the benchmark named `MTEB(eng, v2)`, 41 tasks.
2. **A public leaderboard** on Hugging Face, where model authors submit their results. It is the
   page everybody screenshots.
3. **A Python library, `mteb`**, the reference implementation. It downloads the dataset, runs
   the model, computes the metric, writes a result file. Leaderboard entries *are* those result
   files. When the brief says "measured via MTEB", it means this library ran.

One neighbour to place: **BEIR** (*Benchmarking IR*, Thakur et al., 2021) is the older
collection of 18 **retrieval** datasets in one uniform format — corpus, queries, **qrels** (the
relevance judgments, notebook 0) — that MTEB absorbed as its retrieval task type. SciFact is a
BEIR dataset; the BM25 score of 0.665 that notebooks 0 and 2 quote is from the BEIR paper. In
this project, "MTEB" is the protocol and the library, "BEIR" is the family of datasets.

In [1]:
# --- Configuration: edit, then "Run All" ---
import importlib.util
import os

from llm_testbench.config import find_repo_root

SAMPLE = os.environ.get("LLM_TESTBENCH_SAMPLE", "0") == "1"
ROOT = find_repo_root()  # the notebook runs from notebooks/phase-1/

# `mteb` lives in the `bench` dependency group (uv sync --group bench); the CI does not have it.
MTEB_AVAILABLE = importlib.util.find_spec("mteb") is not None

# The task whose card and data part 3 inspects. The runs of part 4 are always SciFact, the only
# task wired in run_scifact; "NFCorpus" or "FiQA2018" (phase 3 candidates) change part 3 only.
TASK_NAME = "SciFact"
MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # ~20 s; "BAAI/bge-base-en-v1.5" ~2 min
DEVICE = "mps"  # "cpu" on another machine, "cuda" on a GPU
BATCH_SIZE = 64

RUN_EVAL = MTEB_AVAILABLE and not SAMPLE  # part 4.2: MODEL on the real SciFact
RUN_PROMPT_ABLATION = False  # part 4.4, full mode, ~2 min: bge without its query instruction
RUN_BM25 = False  # part 4.4, full mode, ~10 s, needs `uv pip install bm25s PyStemmer`

SHOW_QUERY_INDEX = 0  # the query detailed in part 3.2

# The toy ranking scored by hand in part 4.1: query id -> {document id: similarity}, best first.
# Relevant documents (fixtures): query 1 -> 4983 and 5836; query 3 -> 19238; query 5 -> 13734.
TOY_RUN = {
    "1": {"4983": 0.91, "21389": 0.80, "5836": 0.77, "7912": 0.40},
    "3": {"7912": 0.85, "13734": 0.70, "19238": 0.65},
    "5": {"21389": 0.60, "7912": 0.55, "4983": 0.50},
}

print(f"sample mode: {SAMPLE} — mteb installed: {MTEB_AVAILABLE} — real run: {RUN_EVAL}")

sample mode: True — mteb installed: True — real run: False


In [2]:
import json
import time
import warnings

from llm_testbench.config import Settings
from llm_testbench.eval.embeddings_bench import (
    PUBLISHED_SCIFACT,
    load_report,
    run_scifact,
    save_report,
)
from llm_testbench.eval.loaders import BeirLoader

# torch prints a deprecation warning at import and bm25s draws progress bars: noise for this reading.
warnings.filterwarnings("ignore", category=FutureWarning)
os.environ.setdefault("TQDM_DISABLE", "1")
if MTEB_AVAILABLE:
    import mteb

# The archived report of notebook 2 (make bench-embeddings, 2026-09-03): the numbers we compare to.
RESULTS_PATH = ROOT / "data" / "results" / "embeddings_scifact.json"
archived = load_report(RESULTS_PATH) if RESULTS_PATH.is_file() else None

# The fixtures: six abstracts, three judged queries — the toy example of part 4.1, in both modes.
fixtures = BeirLoader(
    "scifact",
    settings=Settings(fixtures_dir=ROOT / "tests" / "fixtures", data_dir=ROOT / "data" / "cache"),
).load("test")
# The dataset of part 3.2: the fixtures in sample mode, the real SciFact in full mode.
settings = Settings(
    data_dir=ROOT / "data" / "cache",
    fixtures_dir=(ROOT / "tests" / "fixtures") if SAMPLE else None,
    sample=SAMPLE,
)
dataset = BeirLoader("scifact", settings=settings).load("test")

print(
    "archived report:",
    f"{archived.generated_at:%Y-%m-%d}, {len(archived.scores)} models" if archived else "none",
)
print("toy dataset:", fixtures.describe())
print("dataset:", dataset.describe())

archived report: 2026-09-03, 2 models
toy dataset: {'dataset': 'beir/scifact', 'split': 'test', 'documents': 6, 'queries': 3, 'judgments': 4, 'sampled': False}
dataset: {'dataset': 'beir/scifact', 'split': 'test', 'documents': 6, 'queries': 3, 'judgments': 4, 'sampled': False}


## Part 2 — The options: three ways to get a number per model

| Option | What you get | What you do not get | Verdict |
|---|---|---|---|
| **Read the leaderboard** | a free score per task per model, same protocol for every open model | an independent check (entries are submitted; for closed APIs, often by the vendor); your dataset version; any idea of what the code does | the starting point, never the evidence |
| **Our own harness** (phase 2) | metrics on our pgvector index, our chunks, our prompts — the thing we ship | a reference: if our nDCG is 0.62, is the harness wrong or the model? | needed, but not first |
| **Run `mteb` ourselves** on the task we care about | the published protocol, rerun on our machine; a number that can be set next to the leaderboard **and** that the harness must later reproduce | our index: `mteb` embeds the raw corpus in memory and searches exactly, not through pgvector | **chosen in phase 1** (notebook 2, part 5) |

The third option is the only one that gives two cross-checks at once: `mteb`'s number against
the card (does the protocol run correctly here?), then our harness against `mteb`'s number (does
our implementation do what the protocol does?). The phase 1 result — bge-base-en-v1.5 at 0.740
against 0.7404 on its card — closed the first; phase 2 owes the second.

**What we take from MTEB, and what we leave.**

- Taken: the **retrieval** task type on BEIR datasets (SciFact now; NFCorpus and FiQA in phase 3),
  **nDCG@10** as the main score, the text conventions the scores assume, the library as the
  reference implementation.
- Left: the seven other task types (the system classifies and clusters nothing), the leaderboard
  **average** (it ranks models for a use we do not have), full benchmark runs (`MTEB(eng, v2)`:
  41 tasks, hours of GPU), submitting results, and the French benchmark `MTEB(fra, v1)` until a
  French corpus enters the project.

## Part 3 — Guided reading: what `mteb` does when it scores a model on SciFact

The library has one entry point, `mteb.evaluate(model, tasks)`, and three objects to know: a
**task**, a **model** (any object that follows its encoder protocol), a **result**. We follow one
call from the task definition to the result file.

### 3.1 A task is a dataset plus a protocol

`mteb.get_task("SciFact")` returns a `SciFact` object, a subclass of `AbsTaskRetrieval`
([abstasks/retrieval.py#L83](https://github.com/embeddings-benchmark/mteb/blob/2.20.5/mteb/abstasks/retrieval.py#L83)). Its `metadata` is the contract that everybody who
publishes a SciFact score accepted, usually without reading it:

- the **type** (`Retrieval`) picks the evaluator and the metrics;
- the **main score** (`ndcg_at_10`) is the one number that goes to the leaderboard;
- the **eval split** (`test`): the 300 judged queries, never the train split;
- the **dataset** is a Hugging Face repository **pinned to a revision** (`mteb/scifact` at
  `d56462d…`): the data cannot drift under the score;
- the **descriptive statistics** are computed once and shipped with the library: how many
  documents, queries and judgments, how long the texts are. They answer "what does this number
  cover?" before any download;
- `k_values` (`1, 3, 5, 10, 20, 100, 1000`, [abstasks/retrieval.py#L102](https://github.com/embeddings-benchmark/mteb/blob/2.20.5/mteb/abstasks/retrieval.py#L102)) are the
  cut-offs at which every metric is computed; the largest one is how many documents the search
  keeps per query.

The next cell prints them. It needs `mteb` but no network: the metadata is Python code inside
the package.

In [3]:
if MTEB_AVAILABLE:
    task = mteb.get_task(TASK_NAME)
    meta = task.metadata
    print(
        f"{meta.name}: type {meta.type}, main score {meta.main_score}, eval splits {meta.eval_splits}"
    )
    print(
        f"dataset: {meta.dataset['path']} @ {meta.dataset['revision'][:12]} — license {meta.license}, domains {meta.domains}"
    )
    print(f"description: {meta.description}")
    print(f"reference: {meta.reference}")
    print(
        f"k values: {tuple(task.k_values)} → documents kept per query by the search: {max(task.k_values)}"
    )
    for split, stats in meta.descriptive_stats.items():
        rel = stats["relevant_docs_statistics"]
        docs, queries = stats["documents_text_statistics"], stats["queries_text_statistics"]
        print(
            f"\nshipped statistics ({split}): {stats['num_documents']} documents, {stats['num_queries']} queries, "
            f"{rel['num_relevant_docs']} judgments ({rel['average_relevant_docs_per_query']:.2f} per query, "
            f"max {rel['max_relevant_docs_per_query']})"
        )
        print(
            f"  document length: mean {docs['average_text_length']:.0f} chars, max {docs['max_text_length']} — "
            f"query length: mean {queries['average_text_length']:.0f} chars"
        )
else:
    print(
        "mteb is not installed (bench group): task card not shown — the values are in the text above"
    )

SciFact: type Retrieval, main score ndcg_at_10, eval splits ['test']
dataset: mteb/scifact @ d56462d0e63a — license cc-by-nc-4.0, domains ['Academic', 'Medical', 'Written']
description: SciFact verifies scientific claims using evidence from the research literature containing scientific paper abstracts.
reference: https://github.com/allenai/scifact
k values: (1, 3, 5, 10, 20, 100, 1000) → documents kept per query by the search: 1000

shipped statistics (test): 5183 documents, 300 queries, 339 judgments (1.13 per query, max 5)
  document length: mean 1499 chars, max 10127 — query length: mean 90 chars


### 3.2 The same data, loaded twice

The project has had its own SciFact loader since phase 0: [`BeirLoader.load`](../../src/llm_testbench/eval/loaders/beir.py#L55) reads the original BEIR
repositories (`BeIR/scifact` for corpus and queries, `BeIR/scifact-qrels` for the judgments)
and [`build_retrieval_dataset`](../../src/llm_testbench/eval/loaders/beir.py#L107) joins and validates them into a [`RetrievalDataset`](../../src/llm_testbench/eval/types.py#L61). `mteb` reads **its own** copy,
`mteb/scifact`, one repository with three files (`corpus.jsonl`, `queries.jsonl`,
`qrels/test.jsonl`), through `RetrievalDatasetLoader`
([abstasks/retrieval_dataset_loaders.py#L44](https://github.com/embeddings-benchmark/mteb/blob/2.20.5/mteb/abstasks/retrieval_dataset_loaders.py#L44)).

Two copies of the same data are a classic source of silent gaps — a document dropped here, a
query renumbered there — so the next cell checks that they agree: same counts, same identifiers,
same judgments. In sample mode our loader reads the six-document fixtures and the comparison is
skipped (it would download the real corpus); in full mode both sides are the real thing: 5,183
abstracts, 300 queries, 339 judgments.

One rule both loaders share, worth noticing: a query without any relevant document is dropped
before scoring — ours at construction (the validator refuses it), `mteb`'s in
`_filter_queries_without_positives` ([abstasks/retrieval.py#L67](https://github.com/embeddings-benchmark/mteb/blob/2.20.5/mteb/abstasks/retrieval.py#L67)). Notebook 0
explained why: an unjudged query scores zero by construction and drags the average down without
an error.

In [4]:
print("ours:", dataset.describe())
query = dataset.queries[SHOW_QUERY_INDEX]
print(f"\nquery {query.query_id}: {query.text!r}")
for doc_id in sorted(dataset.relevant_doc_ids(query.query_id)):
    doc = dataset.document_by_id(doc_id)
    print(f"  relevant {doc_id}: {doc.title!r} — {doc.text[:110]}…")

if MTEB_AVAILABLE and not SAMPLE and TASK_NAME == "SciFact":
    task.load_data()
    split = task.dataset["default"]["test"]
    corpus, queries, relevant_docs = split["corpus"], split["queries"], split["relevant_docs"]
    judgments = sum(len(judged) for judged in relevant_docs.values())
    same_docs = set(corpus["id"]) == {d.doc_id for d in dataset.documents}
    same_queries = set(queries["id"]) == {q.query_id for q in dataset.queries}
    print(f"\nmteb: {len(corpus)} documents, {len(queries)} queries, {judgments} judgments")
    print(f"mteb columns — corpus {corpus.column_names}, queries {queries.column_names}")
    print(
        f"same document ids: {same_docs} — same query ids: {same_queries} — same judgments: {relevant_docs == dataset.qrels}"
    )
else:
    print("\n(the comparison with mteb's copy runs in full mode, with mteb installed, on SciFact)")

ours: {'dataset': 'beir/scifact', 'split': 'test', 'documents': 6, 'queries': 3, 'judgments': 4, 'sampled': False}

query 1: '0-dimensional biomaterials show inductive properties.'
  relevant 4983: 'Microstructural development of human brain' — Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical dev…
  relevant 5836: 'Induction of myelodysplasia by myeloid-derived suppressor cells' — Myelodysplastic syndromes are age-dependent stem cell malignancies that share biological features of activated…

(the comparison with mteb's copy runs in full mode, with mteb installed, on SciFact)


### 3.3 The protocol, step by step, in the library's own source

`mteb.evaluate` ends up in `AbsTaskRetrieval._evaluate_subset`
([abstasks/retrieval.py#L327](https://github.com/embeddings-benchmark/mteb/blob/2.20.5/mteb/abstasks/retrieval.py#L327)). Six steps; each one is a convention that a published
score silently assumes.

**1. Documents become `title + " " + text`.** `_corpus_to_dict`
([_create_dataloaders.py#L56](https://github.com/embeddings-benchmark/mteb/blob/2.20.5/mteb/_create_dataloaders.py#L56-L72)) builds the text the model sees: the title, a space,
the body — when a title exists. A score computed on the body alone is a score for a different
input. The project embedded chunks without their title until 2026-11-02; [`EmbeddingInput`](../../src/llm_testbench/ingest/chunking.py#L41) now
carries this convention into the ingestion pipeline, so that the phase 2 harness measures the
same thing `mteb` measured.

**2. Queries get the model's instruction, if it has one.** Some models were trained to see a
prefix before a query — bge: `"Represent this sentence for searching relevant passages: "`; e5:
`"query: "`, and `"passage: "` in front of the documents too. `mteb` stores those prefixes in
each model's `ModelMeta` ([models/model_implementations/bge_models.py#L9](https://github.com/embeddings-benchmark/mteb/blob/2.20.5/mteb/models/model_implementations/bge_models.py#L9-L10),
[models/model_implementations/e5_models.py#L31](https://github.com/embeddings-benchmark/mteb/blob/2.20.5/mteb/models/model_implementations/e5_models.py#L31-L34)) and `get_prompt_name`
([models/abs_encoder.py#L386](https://github.com/embeddings-benchmark/mteb/blob/2.20.5/mteb/models/abs_encoder.py#L386)) picks the right one, from the most specific
(`"SciFact-query"`) to the most generic (`"query"`). all-MiniLM-L6-v2 has none. The leaderboard
score of bge is a score *with* the instruction; part 4.4 measures the difference.

**3. Encode, through one protocol.** The model receives batches of `{"text": [...]}` and a
`prompt_type` (`query` or `document`), and returns a matrix: `EncoderProtocol.encode`
([models/models_protocols.py#L86](https://github.com/embeddings-benchmark/mteb/blob/2.20.5/mteb/models/models_protocols.py#L86-L111)). `SentenceTransformerEncoderWrapper`
([models/sentence_transformer_wrapper.py#L240](https://github.com/embeddings-benchmark/mteb/blob/2.20.5/mteb/models/sentence_transformer_wrapper.py#L240)) implements it for every Hugging Face
model; our [`ClientEncoder.encode`](../../src/llm_testbench/eval/embeddings_bench.py#L156) implements it on top of [`LLMClient.embed`](../../src/llm_testbench/llm/client.py#L315), so that an API model goes
through our governance, cost and traces instead of `mteb`'s own OpenAI wrapper.

**4. Search: exact cosine, top 1,000.** `SearchEncoderWrapper`
([models/search_wrappers.py#L253](https://github.com/embeddings-benchmark/mteb/blob/2.20.5/mteb/models/search_wrappers.py#L253-L280)) encodes the whole corpus in memory (50,000
documents at a time), computes `model.similarity` — cosine for every model here — between each
query and each document, and keeps the top `max(k_values)` = 1,000 per query with `torch.topk`.
**No index, no approximation.** Our pgvector HNSW index is approximate by construction
(notebook 2, pitfall 7): a small gap between the harness and `mteb` is expected, and must be
measured rather than explained away.

**5. Score with `pytrec_eval`.** `calculate_retrieval_scores`
([_evaluators/retrieval_metrics.py#L580](https://github.com/embeddings-benchmark/mteb/blob/2.20.5/mteb/_evaluators/retrieval_metrics.py#L580-L602)) hands the rankings and the qrels to
`pytrec_eval`, the Python binding of `trec_eval` — the tool the information-retrieval community
has scored with since the 1990s — and asks for `ndcg_cut`, `map_cut`, `recall`, `P` and
`success` at every k; MRR is computed in Python next to it. Per query, then averaged.
**nDCG@10** is the main score because it uses the rank and the relevance grade, which recall@10
ignores; part 4.1 defines it for real.

**6. The result is a file.** A `TaskResult` — task name, dataset revision, `mteb` version, time
per phase, 150+ metric keys — written by `ResultCache`
([cache/result_cache.py#L206](https://github.com/embeddings-benchmark/mteb/blob/2.20.5/mteb/cache/result_cache.py#L206)) under `~/.cache/mteb/results/<model>/<revision>/SciFact.json`.
The leaderboard is a collection of such files. With the default `overwrite_strategy`
(`"only-missing"`, [evaluate.py#L49](https://github.com/embeddings-benchmark/mteb/blob/2.20.5/mteb/evaluate.py#L49-L63)) a second call returns the file instead of
rerunning; [`run_scifact`](../../src/llm_testbench/eval/embeddings_bench.py#L167) passes `"always"`, otherwise its "time" column would measure a cache
hit.

The next cell shows step 2's inputs for the models of notebook 2, plus e5 for contrast — from
the metadata only, nothing is loaded.

In [5]:
MODELS = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "BAAI/bge-base-en-v1.5",
    "intfloat/e5-base-v2",
    "openai/text-embedding-3-small",
]
if MTEB_AVAILABLE:
    for name in MODELS:
        card = mteb.get_model_meta(name)
        prompts = (card.loader_kwargs or {}).get("model_prompts") or {}
        shown = {k: v for k, v in prompts.items() if k in ("query", "document")} or "none"
        declared = card.training_datasets
        similarity = getattr(card.similarity_fn_name, "value", card.similarity_fn_name)
        print(name)
        window = int(card.max_tokens) if card.max_tokens else "?"
        print(
            f"  {card.embed_dim} dimensions — window {window} tokens — similarity {similarity} — open weights {card.open_weights}"
        )
        print(f"  prompts: {shown}")
        print(
            f"  declared training sets: {'not disclosed' if declared is None else len(declared)} — "
            f"zero-shot on {TASK_NAME}: {card.is_zero_shot_on([task])}"
        )
else:
    print("mteb is not installed: the prompts are quoted in the text above")

sentence-transformers/all-MiniLM-L6-v2
  384 dimensions — window 256 tokens — similarity cosine — open weights True
  prompts: none
  declared training sets: 10 — zero-shot on SciFact: True
BAAI/bge-base-en-v1.5
  768 dimensions — window 512 tokens — similarity cosine — open weights True
  prompts: {'query': 'Represent this sentence for searching relevant passages: '}
  declared training sets: 6 — zero-shot on SciFact: True
intfloat/e5-base-v2
  768 dimensions — window 512 tokens — similarity cosine — open weights True
  prompts: {'query': 'query: ', 'document': 'passage: '}
  declared training sets: 10 — zero-shot on SciFact: True
openai/text-embedding-3-small
  1536 dimensions — window 8191 tokens — similarity cosine — open weights False
  prompts: none
  declared training sets: not disclosed — zero-shot on SciFact: None


### 3.4 What `src/` adds around the library

[`run_scifact`](../../src/llm_testbench/eval/embeddings_bench.py#L167) is forty lines: for each model name, load it (`mteb.get_model` for a Hugging Face
name, [`ClientEncoder`](../../src/llm_testbench/eval/embeddings_bench.py#L95) for a `backend/model` name of our registry), call `mteb.evaluate` on the
task with `overwrite_strategy="always"`, read the scores of the test split, and fill a
[`ModelScore`](../../src/llm_testbench/eval/embeddings_bench.py#L45): nDCG@10, recall@10, MAP@10, dimensions, wall time, API cost, and the published
score with its source. The published scores live in the `PUBLISHED_SCIFACT` table at the top of
the module ([embeddings_bench.py](../../src/llm_testbench/eval/embeddings_bench.py)), each with
where it was read and when; `None` where no reference exists, said as is. A [`BenchmarkReport`](../../src/llm_testbench/eval/embeddings_bench.py#L67) groups the
rows, [`save_report`](../../src/llm_testbench/eval/embeddings_bench.py#L234) archives it as JSON in `data/results/`, and `make bench-embeddings` is the
one command behind notebook 2's table.

Why wrap at all, rather than call `mteb` from a notebook? Three reasons that matter again in
phase 2: the API model must go through **our** client (data class, cost, traces — ADR 006); the
numbers must land in a **versioned, typed** file next to their reference, not in a cache under
the home directory; and the cross-check with the literature must be a **line of code**, not a
habit.

The adapter is the part worth reading: a class with an `encode(inputs, **kwargs)` method and an
`mteb_model_meta` attribute is enough for `mteb` to treat it as a model ([`ClientEncoder._model_meta`](../../src/llm_testbench/eval/embeddings_bench.py#L119) builds
that metadata). It accumulates the cost of every batch: the "cost" column of the table is "what
indexing SciFact with this API model costs".

## Part 4 — Run

### 4.1 nDCG@10 by hand, then by `mteb`'s scorer

Before trusting a 0.740, compute one by hand. The fixtures of phase 0 are the toy: six
abstracts, three judged queries — query 1 has two relevant documents (4983 and 5836), query 3
one (19238), query 5 one (13734). `TOY_RUN`, at the top, is an imaginary ranking returned by an
imaginary model: for each query, document ids with a similarity score, best first.

Vocabulary, in the order the metric needs it:

- the **rank** of a document is its position in the ranking, 1 for the best score;
- its **gain** is its relevance grade in the qrels: 1 if relevant, 0 otherwise (SciFact is
  binary; NFCorpus in phase 3 has grades 0 to 2);
- the gain is **discounted** by the rank, divided by log₂(rank + 1): a relevant document at
  rank 1 counts 1, at rank 2 it counts 0.63, at rank 3 0.5, at rank 10 0.29. That is the
  "placed at the top" of the earlier definitions, made precise;
- **DCG@10** (discounted cumulative gain) sums the discounted gains of the first ten results;
- **IDCG@10** is the DCG of the *ideal* ranking, every relevant document first;
- **nDCG@10** = DCG@10 / IDCG@10: 1 is perfect, 0 means nothing relevant in the top ten. Then
  the mean over the queries.

For `TOY_RUN` as shipped:

| query | relevant found at ranks | DCG@10 | IDCG@10 | nDCG@10 |
|---|---|---|---|---|
| 1 | 1 and 3 (a distractor at rank 2) | 1/log₂2 + 1/log₂4 = 1 + 0.5 = 1.5 | 1 + 1/log₂3 = 1.631 | 0.920 |
| 3 | 3 | 1/log₂4 = 0.5 | 1 | 0.500 |
| 5 | not in the top ten | 0 | 1 | 0.000 |
| **mean** | | | | **0.473** |

The next cell feeds the same ranking and the fixtures' qrels to `calculate_retrieval_scores`,
the function `mteb` calls at step 5 — the same `pytrec_eval` computation behind every SciFact
number on the leaderboard — and prints the per-query values next to the mean. Edit `TOY_RUN`
(swap two documents, add the missing one for query 5) and rerun: the metric must move the way
the table predicts.

In [6]:
if MTEB_AVAILABLE:
    from mteb._evaluators.retrieval_metrics import calculate_retrieval_scores

    scored = calculate_retrieval_scores(TOY_RUN, fixtures.qrels, [1, 3, 5, 10])
    for query_id, per_query in scored.all_scores.items():
        ranked = sorted(TOY_RUN[query_id], key=TOY_RUN[query_id].get, reverse=True)
        hits = [
            rank
            for rank, doc_id in enumerate(ranked, start=1)
            if doc_id in fixtures.qrels[query_id]
        ]
        print(
            f"query {query_id}: ranking {ranked} — relevant at ranks {hits} — "
            f"nDCG@10 {per_query['ndcg_cut_10']:.3f}, recall@10 {per_query['recall_10']:.2f}"
        )
    print(
        f"\nmean over the queries: nDCG@10 {scored.ndcg['NDCG@10']:.3f} — recall@10 {scored.recall['Recall@10']:.3f} — "
        f"MRR@10 {scored.mrr['MRR@10']:.3f} — precision@10 {scored.precision['P@10']:.3f}"
    )
else:
    print("mteb is not installed: the hand computation above stands alone (mean nDCG@10 = 0.473)")

query 1: ranking ['4983', '21389', '5836', '7912'] — relevant at ranks [1, 3] — nDCG@10 0.920, recall@10 1.00
query 3: ranking ['7912', '13734', '19238'] — relevant at ranks [3] — nDCG@10 0.500, recall@10 1.00
query 5: ranking ['21389', '7912', '4983'] — relevant at ranks [] — nDCG@10 0.000, recall@10 0.00

mean over the queries: nDCG@10 0.473 — recall@10 0.667 — MRR@10 0.444 — precision@10 0.100


### 4.2 The real run, through `run_scifact`

In full mode with `mteb` installed, the next cell runs `MODEL` on the real SciFact exactly as
`make bench-embeddings` does, and sets the result next to the archived row of the same model.
Two runs, weeks apart, must give the same nDCG@10 to the fifth decimal: the protocol is
deterministic once the model, the data revision and the text conventions are fixed. The time is
not — it depends on the machine, the device and the batch size. In sample mode, or without
`mteb`, the cell prints the archived report instead.

Nothing scrolls during the run (`run_scifact` silences the progress bar); about 20 s for
all-MiniLM-L6-v2 on Apple Silicon, then the table with the gap to the published score where one
is recorded.

In [7]:
if RUN_EVAL:
    started = time.perf_counter()
    report = run_scifact([MODEL], device=DEVICE, batch_size=BATCH_SIZE)
    print(report.table())
    print(f"\nwall time {time.perf_counter() - started:.0f} s")
    previous = next((s for s in archived.scores if s.model == MODEL), None) if archived else None
    if previous is not None:
        print(
            f"archived {archived.generated_at:%Y-%m-%d}: nDCG@10 {previous.ndcg_at_10:.5f} — today {report.scores[0].ndcg_at_10:.5f} "
            f"— difference {report.scores[0].ndcg_at_10 - previous.ndcg_at_10:+.5f}"
        )
elif archived is not None:
    print(f"archived report of {archived.generated_at:%Y-%m-%d} (make bench-embeddings):\n")
    print(archived.table())
else:
    print("no run and no archived report: run `make bench-embeddings` once")

archived report of 2026-09-03 (make bench-embeddings):

model                                       nDCG@10 published     gap   dim    time      cost
sentence-transformers/all-MiniLM-L6-v2        0.645         —       —   384     20s     local
BAAI/bge-base-en-v1.5                         0.740     0.740  -0.000   768    120s     local


In [8]:
if MTEB_AVAILABLE and not SAMPLE:
    revision = mteb.get_model_meta(MODEL).revision
    result_file = (
        mteb.ResultCache().cache_path
        / "results"
        / MODEL.replace("/", "__")
        / revision
        / "SciFact.json"
    )
    if result_file.is_file():
        raw = json.loads(result_file.read_text())
        print(f"{result_file}\n")
        print(
            f"mteb {raw['mteb_version']} — dataset revision {raw['dataset_revision'][:12]} — evaluation {raw['evaluation_time']:.0f} s"
        )
        for phase in raw["evaluation_phases"]:
            print(f"  {phase['name']:<34} {phase['end'] - phase['start']:7.1f} s")
        scores = raw["scores"]["test"][0]
        shown_keys = [
            "main_score",
            "ndcg_at_10",
            "ndcg_at_100",
            "recall_at_10",
            "recall_at_100",
            "precision_at_10",
            "map_at_10",
            "mrr_at_10",
            "nauc_ndcg_at_10_max",
        ]
        print(f"\n{len(scores)} keys in the result; a few of them:")
        for key in shown_keys:
            print(f"  {key:<22} {scores[key]}")
    else:
        print(f"no cached result for {MODEL}: run the previous cell in full mode first")
else:
    print("the raw result file is shown in full mode with mteb installed")

the raw result file is shown in full mode with mteb installed


### 4.3 Against the published score — and what "published" means

Three places publish a SciFact number for a model, and they are not equally trustworthy:

1. **The paper or the model card.** bge's card lists its MTEB results per task; that is the
   0.7404 in `PUBLISHED_SCIFACT`, read on 2026-09-03. all-MiniLM-L6-v2's card has no SciFact
   line, so our table says `—` rather than guessing.
2. **The leaderboard.** Its entries come from result files in a public repository
   (`embeddings-benchmark/results`) that `mteb` can download (`ResultCache().load_results`); the
   whole repository weighs hundreds of megabytes for a handful of numbers, so this notebook does
   not pull it. For a closed API model, the entry was produced by whoever ran it — often the
   vendor.
3. **The BEIR paper** (2021) for BM25, the lexical baseline: 0.665, produced with Anserini (a
   Lucene wrapper) and its tokenization. Another BM25 implementation gives another number; part
   4.4 measures one.

One flag the leaderboard shows next to each score: **zero-shot**. `mteb` asks every model's
metadata which datasets it was trained on (`training_datasets`,
[models/model_implementations/bge_models.py#L402](https://github.com/embeddings-benchmark/mteb/blob/2.20.5/mteb/models/model_implementations/bge_models.py#L402)) and marks a score as zero-shot when
the task's data is not among them. It is a **declaration**, maintained by hand in the library:
bge declares six MTEB datasets, MiniLM ten, OpenAI nothing (`None`: not disclosed, so no flag
either way). Contamination — test data inside the training set — is treated as a measurement
topic in phase 2 (notebook `phase-2/03`), not as a checkbox.

In [9]:
print("published references recorded in src/ (PUBLISHED_SCIFACT):")
for name, (value, source) in PUBLISHED_SCIFACT.items():
    shown = f"{value:.4f}" if value is not None else "   —  "
    print(f"  {shown}  {name:<42} {source}")

if archived is not None:
    print(f"\ngap of the archived scores ({archived.generated_at:%Y-%m-%d}) to their reference:")
    for score in archived.scores:
        gap = (
            f"{score.gap_to_published:+.4f}"
            if score.gap_to_published is not None
            else "no reference"
        )
        print(f"  {score.model:<42} measured {score.ndcg_at_10:.4f} — {gap}")

published references recorded in src/ (PUBLISHED_SCIFACT):
  0.6650  BM25 (Anserini)                            Thakur et al. 2021, BEIR, table 2
  0.6650  mteb/baseline-bm25s                        Thakur et al. 2021, BEIR, table 2 — Anserini BM25, not bm25s
  0.7404  BAAI/bge-base-en-v1.5                      HF model card, MTEB results, read on 2026-09-03
  0.7194  intfloat/e5-base-v2                        HF model card, MTEB results, read on 2026-09-03
     —    sentence-transformers/all-MiniLM-L6-v2     no SciFact score on the HF card
     —    openai/text-embedding-3-small              OpenAI does not publish per-task scores

gap of the archived scores (2026-09-03) to their reference:
  sentence-transformers/all-MiniLM-L6-v2     measured 0.6451 — no reference
  BAAI/bge-base-en-v1.5                      measured 0.7404 — -0.0000


### 4.4 Two experiments the protocol invites

Both are off by default (full mode, `mteb` installed); the numbers measured on 2026-11-04 are in
part 6.

**The query instruction, removed.** `mteb.get_model(..., model_prompts={"query": "",
"document": ""})` loads bge-base-en-v1.5 without its prefix, and `mteb.evaluate` runs it: the
same weights, evaluated the way a tutorial that forgets the instruction evaluates them. Set
`RUN_PROMPT_ABLATION = True` (about two minutes; `mteb` warns that the prompts are overridden,
which is the point). The run writes to a separate result cache under `data/results/`, which
leaves the official result in `~/.cache/mteb` alone and archives the raw result file in the
repository — under an `experiments/` folder, because `mteb` notices that the run deviates from
the canonical model, next to a `run_settings.jsonl` that records the library versions. This is
the one place where the notebook drives the library directly, outside `src/`: an ablation, not
a code path of the project.

**The lexical baseline, measured here.** `mteb` ships BM25 as a model (`mteb/baseline-bm25s`,
built on the `bm25s` library: English stop words, Snowball stemmer, k1 = 1.5, b = 0.75 —
[models/model_implementations/bm25.py#L589](https://github.com/embeddings-benchmark/mteb/blob/2.20.5/mteb/models/model_implementations/bm25.py#L589)). It goes through our [`run_scifact`](../../src/llm_testbench/eval/embeddings_bench.py#L167) like
any other name, in seconds, and the report is archived in `data/results/bm25_scifact.json`. It
needs two packages that are not in the `bench` group yet — `uv pip install bm25s PyStemmer`
into `.venv` — then `RUN_BM25 = True`. The number is not the BEIR paper's: different
implementation, different tokenization; measuring it is the point.

In [10]:
if RUN_PROMPT_ABLATION and MTEB_AVAILABLE and not SAMPLE:
    scifact = mteb.get_task("SciFact")
    ablation_cache = mteb.ResultCache(ROOT / "data" / "results" / "mteb_prompt_ablation")
    bare = mteb.get_model(
        "BAAI/bge-base-en-v1.5", device=DEVICE, model_prompts={"query": "", "document": ""}
    )
    started = time.perf_counter()
    bare_result = mteb.evaluate(
        bare,
        [scifact],
        encode_kwargs={"batch_size": BATCH_SIZE},
        show_progress_bar=False,
        cache=ablation_cache,
        overwrite_strategy="always",
    )
    bare_scores = bare_result.task_results[0].scores["test"][0]
    print(
        f"bge-base-en-v1.5 without its query instruction: nDCG@10 {bare_scores['ndcg_at_10']:.4f}, "
        f"recall@10 {bare_scores['recall_at_10']:.4f} ({time.perf_counter() - started:.0f} s)"
    )
    with_prompt = (
        next((s for s in archived.scores if s.model == "BAAI/bge-base-en-v1.5"), None)
        if archived
        else None
    )
    if with_prompt is not None:
        print(
            f"with the instruction (archived report): nDCG@10 {with_prompt.ndcg_at_10:.4f} — "
            f"difference {bare_scores['ndcg_at_10'] - with_prompt.ndcg_at_10:+.4f}"
        )
    print(f"raw result archived under {ablation_cache.cache_path.relative_to(ROOT)}/")
else:
    print("prompt ablation skipped (RUN_PROMPT_ABLATION = True, full mode, mteb installed)")

prompt ablation skipped (RUN_PROMPT_ABLATION = True, full mode, mteb installed)


In [11]:
BM25_PATH = ROOT / "data" / "results" / "bm25_scifact.json"
if RUN_BM25 and MTEB_AVAILABLE and not SAMPLE:
    if importlib.util.find_spec("bm25s") is None:
        print(
            "bm25s is not installed: `uv pip install bm25s PyStemmer` (into .venv), then rerun this cell"
        )
    else:
        bm25_report = run_scifact(["mteb/baseline-bm25s"], device=None)
        save_report(bm25_report, BM25_PATH)
        print(bm25_report.table())
        print(f"\n→ {BM25_PATH.relative_to(ROOT)}")
elif BM25_PATH.is_file():
    bm25_report = load_report(BM25_PATH)
    print(f"archived BM25 report of {bm25_report.generated_at:%Y-%m-%d}:\n")
    print(bm25_report.table())
else:
    print("BM25 skipped (RUN_BM25 = True, full mode, mteb and bm25s installed)")

archived BM25 report of 2026-11-04:

model                                       nDCG@10 published     gap   dim    time      cost
mteb/baseline-bm25s                           0.686     0.665  +0.021     ?      6s     local


## Pitfalls

1. **Two APIs, one name.** `mteb` 2.x (`mteb.get_task`, `mteb.evaluate`) broke with 1.x
   (`MTEB(tasks=...).run(model)`), and most tutorials, blog posts and model READMEs still show
   1.x. A snippet that does not run is not a bug in your environment.
2. **A model without its instruction is another model.** bge evaluated without its query prefix
   loses three thousandths the card never lost (part 6); e5 without `query: ` and `passage: `
   loses more, its authors say — the prefixes are part of the weights. In production the same
   prefix must be applied to the question at query time, and *only* to the question: the index
   carries the documents' convention, the request carries the query's.
3. **The title is part of the input.** `title + " " + text` is what every SciFact score assumes
   (step 1). Chunks embedded without their title, as the pipeline did until 2026-11-02, give a
   number no published score can be compared to.
4. **Silent truncation.** all-MiniLM-L6-v2 reads 256 tokens, bge 512; a SciFact abstract
   averages 1,500 characters, roughly 350 tokens, with a maximum above 10,000. MiniLM scores the
   first two thirds of most abstracts and never says so. That is part of its 0.645.
5. **The result cache answers instead of the model.** With the default
   `overwrite_strategy="only-missing"`, a second `mteb.evaluate` returns the JSON on disk: the
   right number, in zero seconds. Any timing you report must come from `"always"` (what
   `run_scifact` does) or from a fresh cache.
6. **Exact search here, approximate search there.** `mteb` compares every query to every
   document; pgvector's HNSW does not. The phase 2 cross-check must separate "our metric is
   wrong" from "our index misses neighbours" — the second is tuned with `ef_search` and measured
   as recall against the exact search.
7. **300 queries is a small sample.** One point of nDCG@10 between two models on SciFact is
   within the noise of the query sample; phase 2 adds bootstrap confidence intervals before any
   "A beats B".
8. **The average ranks models for someone else's use.** The leaderboard average mixes 41 tasks;
   a model can lead it and lose on the one retrieval task that looks like yours. Read the
   column, not the rank — then measure on your golden set (phase 2.4).
9. **"Zero-shot" is a declaration.** The flag depends on the training data the model's authors
   listed in `mteb`'s metadata, sometimes incomplete (`# not in mteb` comments in the source). A
   vendor that discloses nothing gets no flag, not a bad one.
10. **Closed models drift under a fixed name.** `openai/text-embedding-3-small` is "revision 3"
    in `mteb`'s metadata because nothing better exists: a leaderboard entry for an API model is
    a snapshot of that API on that day.

## Part 6 — The numbers

SciFact, test split, nDCG@10, `mteb` 2.20.5, all measured on the same Apple Silicon machine
(`mps`; BM25 on CPU):

| arm | nDCG@10 | recall@10 | published | gap | time | measured on |
|---|---|---|---|---|---|---|
| BM25 — `mteb/baseline-bm25s` (bm25s, stemmed) | **0.686** | 0.819 | 0.665 (BEIR 2021, Anserini) | +0.021 | 6 s | 2026-11-04 |
| all-MiniLM-L6-v2 | 0.645 | 0.783 | — | — | 20 s | 2026-09-03 |
| bge-base-en-v1.5 | **0.740** | 0.874 | 0.7404 (model card) | 0.000 | 120 s | 2026-09-03 |
| bge-base-en-v1.5 **without** its query instruction | 0.738 | 0.866 | — | −0.003 vs 0.740 | 122 s | 2026-11-04 |

Archived in `data/results/`: `embeddings_scifact.json` (notebook 2), `bm25_scifact.json`, and
the raw `mteb` result of the ablation under `mteb_prompt_ablation/`.

What the table says:

- **The lexical baseline, measured here, is two points above the paper's.** Different tokenizer
  (Snowball stemming, stop words) and parameters (k1 1.5 and b 0.75 instead of Anserini's 0.9
  and 0.4). "BM25" is a family, not a number: the baseline the phase 3 hybrid search has to
  beat is the one we can rerun, 0.686, not the one we read.
- **MiniLM, the tutorial model, is four points below that baseline**, not two: the gap notebook
  2 announced grows once the baseline is measured on the same machine. Its recall@10 is also
  lower (0.783 against 0.819): it finds the relevant abstract less often, not only
  lower in the list.
- **bge-base-en-v1.5 reproduces its card exactly**, and the ablation says how much of its score
  the 57-character instruction carries: 0.003 of nDCG@10 (0.738 without it
  against 0.740 with it; recall@10 0.866 against 0.874). Small on this model
  and dataset — the v1.5 release made the instruction optional — but free to get right, and the
  e5 authors state that their `query: ` and `passage: ` prefixes are required, not optional.

## Part 7 — What the roadmap takes from MTEB, phase by phase

| Phase | What MTEB provides | What we do with it |
|---|---|---|
| **1** (done) | the protocol and the library | three models measured on SciFact through [`run_scifact`](../../src/llm_testbench/eval/embeddings_bench.py#L167); bge's card reproduced; the API model through our client |
| **2** | the numbers to reproduce: 0.740, 0.645, BM25 0.686 | our harness (recall@k, MRR, nDCG, on **our** pgvector index) must find them again, with the same text convention and the same query prefix; the difference between exact and HNSW search measured as such |
| **3** | the BEIR datasets with graded judgments (NFCorpus) and scale (FiQA); the reranking task type | hybrid search and re-ranking measured against the BM25 and dense baselines above; graded nDCG where the qrels are 0 to 2 |
| **5** | nothing new | the embedding model is a production dependency: a drift check compares `mteb`'s number before and after an upgrade |
| **6** | the reference number for a model we serve ourselves | an embedding model served by vLLM, or quantized, is accepted when `mteb` gives it the same nDCG@10 as the Hugging Face weights; otherwise the serving is wrong |

Not needed, and why: the seven non-retrieval task types (the system classifies and clusters
nothing); the leaderboard average (it ranks for a generic use); full benchmark runs (hours of GPU
for columns we do not read); result submission (the testbench publishes its numbers in
`data/results/`, with their source); `MTEB(fra, v1)` (25 tasks, 5 of them retrieval) until a
French corpus enters the project — the day it does, the protocol is the same and the library
already has it.

## Your turn

1. **`TOY_RUN`** — move document 13734 into the ranking of query 5; swap the scores of 4983 and
   21389 for query 1. Predict the nDCG@10 from the table in 4.1, then rerun 4.1.
2. **`MODEL = "BAAI/bge-base-en-v1.5"`** (full mode, 2 min) — the archived 0.74039 again, to
   the fifth decimal? Look at the phase timings of the raw result in 4.2: where do the two
   minutes go?
3. **`RUN_PROMPT_ABLATION = True`** (full mode, 2 min) — the measured cost of forgetting the
   instruction, on your machine.
4. **`RUN_BM25 = True`** (full mode, `uv pip install bm25s PyStemmer`) — the lexical baseline
   here; then compare its recall@10 with MiniLM's: which one finds the relevant abstract more
   often, which one ranks it higher?
5. **`TASK_NAME = "NFCorpus"`** — the card of a graded dataset in 3.1 (3.6k documents, qrels 0
   to 2, 134k judgments), before any download; part 4 keeps running SciFact.
6. **`SHOW_QUERY_INDEX = 1, 2`** — other claims and the abstracts judged to support or refute
   them: notice that the query is a statement, not a question.

## Interview questions

<details><summary><b>1. What is MTEB, and what does a leaderboard score mean?</b></summary>

Three things: a collection of tasks (retrieval, classification, clustering, STS…, each a
dataset with a main metric), a public leaderboard, and the `mteb` library that produces the
entries. A score is the main metric of one task, computed by the library on a pinned revision
of the dataset, with the model's declared prompts, by whoever submitted it. The average mixes
task types you may not have; the per-task column is the number to read, and rerunning the
library on that task is the way to trust it.
</details>

<details><summary><b>2. Two teams report different SciFact nDCG@10 for the same open model. Usual causes?</b></summary>

In rough order of frequency: the query instruction missing on one side; documents embedded
without their title; a different maximum sequence length (truncation); an approximate index
(HNSW) against exact search; a cached result taken for a run; a different dataset revision or
`mteb` version (the metric code changed between 1.x and 2.x); a different cut-off or similarity
function. All of them are conventions, none of them is the model.
</details>

<details><summary><b>3. Why is nDCG@10 the main score rather than recall@10?</b></summary>

Recall@10 counts the relevant documents found in the first ten, whatever their position, and
ignores the grade. nDCG@10 discounts each one by its rank and weights it by its grade, so it
rewards putting the right document first — which is what a RAG prompt needs, since the first
positions are what the generator reads — and it works on graded datasets (NFCorpus). Report
both: recall says whether the document is there at all (a re-ranker can fix the order, not the
absence), nDCG says how well it is ordered.
</details>

<details><summary><b>4. How does BEIR relate to MTEB?</b></summary>

BEIR (2021) assembled 18 retrieval datasets in one format — corpus, queries, qrels — to evaluate
retrieval zero-shot, with BM25 as the baseline. MTEB (2022) took them as its retrieval task
type, and the `mteb` library replaced the `beir` package as the runner. Our SciFact comes from
the original BEIR repositories; `mteb`'s copy is the same data (checked in 3.2). The BEIR
paper's BM25 numbers remain the lexical reference everybody quotes — and part 4.4 shows that
another BM25 implementation does not reproduce them exactly.
</details>

<details><summary><b>5. Is bge-base-en-v1.5's 0.740 on SciFact a zero-shot number?</b></summary>

According to the training data declared in `mteb`'s metadata, yes: SciFact is not among the six
datasets listed. But it is a declaration maintained by hand, the BEIR training splits are
public, and the model's full training mix is not in the library. Contamination is measured,
not assumed: phase 2 compares scores on a dataset the model cannot have seen. For a closed API
model nothing is declared, so nothing can be said either way.
</details>

<details><summary><b>6. How would you choose an embedding model for a bank's internal documents?</b></summary>

Not from the leaderboard average. Find the two or three MTEB tasks closest to the use (domain,
query shape, language), rerun `mteb` on them for a short list of open and API models, and put
the constraints before the quality column: where the API runs and what it retains (data
governance, ADR 006), the cost and latency of self-hosting (phase 6), the window against the
chunk size. Then measure on an in-house golden set (phase 2.4) with confidence intervals, and
take the cheapest model within the noise of the best one.
</details>

## What we did not do, and why

- **No full benchmark run**: `MTEB(eng, v2)` is 41 tasks and hours of GPU for columns the
  project does not read. One retrieval task, rerun exactly, is the testbench's use of MTEB.
- **No leaderboard download**: the results repository weighs hundreds of megabytes; the
  references we need are copied by hand into `PUBLISHED_SCIFACT`, with their source and date.
- **No API model here**: `make bench-embeddings` adds text-embedding-3-small through our client
  with a key; the protocol is the same and notebook 2 covers it.
- **No in-house nDCG**: the metric is computed by `pytrec_eval` through `mteb` on purpose; our
  own implementation is the phase 2 harness, and this number is what it must reproduce.
- **No reranking task**: cross-encoders arrive in phase 3, where `mteb`'s reranking tasks become
  the reference the same way.
- **`bm25s` not added to the `bench` group**: installed ad hoc for the baseline. Whether BM25
  enters the project through this library or through Postgres full-text search is a phase 2
  plan decision; the dependency follows it.
- **No French benchmark**: `MTEB(fra, v1)` exists in the library; it enters with a French corpus.

**Next**: phase 2, where the harness takes these numbers as its acceptance test.

## Summary glossary

| Term | Definition |
|---|---|
| **MTEB** | Massive Text Embedding Benchmark: a collection of tasks, a public leaderboard, and the `mteb` library that produces the scores. |
| **BEIR** | Collection of retrieval datasets (corpus, queries, qrels) in one format; MTEB's retrieval task type. SciFact is one of them. |
| **Task** | One dataset plus a protocol: a type, a main metric, an evaluation split, a pinned data revision. |
| **Benchmark (named)** | A fixed list of tasks, such as `MTEB(eng, v2)` (41 tasks) or `BEIR` (15); what a leaderboard average is computed on. |
| **Main score** | The one metric of a task that goes to the leaderboard; `ndcg_at_10` for retrieval. |
| **Qrels** | Relevance judgments: for each query, the relevant documents and their grade. The ground truth of every metric here. |
| **Run / ranking** | What a model returns for a query: documents with scores, best first. The input of the metrics. |
| **Rank, gain, discount** | Position in the ranking; relevance grade of the document at that position; division of the gain by log₂(rank + 1). |
| **DCG@10, IDCG@10, nDCG@10** | Sum of discounted gains over the first ten results; the same for the ideal ranking; their ratio, between 0 and 1. |
| **Recall@k, MRR@k, MAP@k** | Share of relevant documents found in the first k; inverse rank of the first relevant one; mean precision over the relevant positions. |
| **`pytrec_eval`** | Python binding of `trec_eval`, the reference scorer of information retrieval; what `mteb` computes the metrics with. |
| **Instruction / prompt** | A prefix some models expect before a query (or a document); part of the model, stored in `mteb`'s metadata. |
| **Encoder protocol** | The interface `mteb` expects from a model: an `encode(inputs, prompt_type, ...)` method and an `mteb_model_meta` attribute. |
| **Exact search** | Similarity of every query with every document, then top-k; what `mteb` does. Opposed to approximate search (HNSW in pgvector). |
| **`ResultCache` / `TaskResult`** | Where `mteb` writes a result file and the file itself: task, data revision, library version, timings, metrics at every k. |
| **Revision** | A pinned version of a dataset or a model on Hugging Face; the reason two runs of the same name compare at all. |
| **Zero-shot** | Scored on a task whose data the model declares it was not trained on; a declaration, not a verification. |
| **Contamination** | Test data present in a model's training set; inflates scores silently; measured in phase 2. |
| **BM25 / Anserini / `bm25s`** | The classic keyword ranking function; the Lucene toolkit the BEIR paper used; the Python library `mteb` ships it with. |
| **Stemming** | Reducing words to a root (`treated` → `treat`) before indexing; part of why two BM25 scores differ. |